# Programvaretesting

Programmer skal være feilfrie og oppføre seg som forventet – lettere sagt enn gjort. Derfor tester vi før de tas i bruk. I 1.8 så vi at testing kan ta opptil halvparten av tiden i et utviklingsprosjekt.

De vanligste testnivåene:

1. **Enhetstesting** – hver enhet (funksjon/klasse) testes isolert.
2. **Integrasjonstesting** – kommuniserer enhetene riktig med hverandre?
3. **Systemtesting** – oppfyller hele systemet kravspesifikasjonen?
4. **Akseptansetesting** – brukerne tester i sitt eget miljø. Dekker systemet behovene deres? Utviklere og brukere kan tolke krav ulikt, så det er viktig å snakke sammen underveis.

Det finnes også tester for ytelse, sikkerhet, kompatibilitet og mer.

Vi bør ha en plan for testingen, og jo mer som kan **automatiseres**, desto bedre. Enhetstesting hindrer tilfeldig testing. Å teste grafiske brukergrensesnitt er vanskeligere, så foreløpig holder vi oss til enhetstester.

Enhetstesting slik vi kjenner det kom på slutten av 1990-tallet med JUnit for Java. Python har `unittest`, som ligner, men vi bruker **pytest**: gratis, lett å lære, krever mindre kode og gir detaljert utskrift.

> **Slik bruker du denne notebooken:** Du skal ikke kjøre noe her. Les forklaringene, lag filene i VS Code og kopier inn koden fra kodeboksene. Testene kjører du i VS Code. Under hvert steg står det hva du skal se når du kjører testene.

## 0 Oppsett

Dette gjør du én gang.

1. Åpne prosjektmappa di i VS Code (*File → Open Folder*).
2. Sjekk at riktig Python-miljø er valgt: `Ctrl+Shift+P` (Mac: `Cmd+Shift+P`) → *Python: Select Interpreter*. Velg `.venv` hvis du har et virtuelt miljø.
3. Åpne en terminal (*Terminal → New Terminal*) og installer pytest:

```
python -m pip install -U pytest
```

   Står det at `python` ikke finnes (vanlig på Mac uten venv), bruker du `python3` i stedet – også i kommandoene lenger ned.

4. Klikk på kolben (*Testing*) i aktivitetslinja til venstre → *Configure Python Tests* → **pytest** → **`.` (Root directory)**. Da finner VS Code testene i alle mappene vi lager.

### Slik kjører du testene

* **Testing-panelet (kolben):** ▷ øverst kjører alle testene, ▷ ved en enkelt test kjører bare den. Grønt ✓ = passerer, rødt ✗ = feiler. Dukker ikke nye tester opp, klikk ↻ (*Refresh Tests*).
* **Test Results** (nederste panel, ved siden av *Terminal*) viser hele utskriften fra pytest. Du kommer også dit ved å klikke på en rød ✗. I eldre versjoner av VS Code heter det *Output → Python Test Log*.
* **Terminalen:** `python -m pytest demo_tests` kjører testene i mappa `demo_tests`. Legg til `-v` for å få én linje per test.

Mappene `.vscode`, `.pytest_cache` og `__pycache__` lager VS Code og Python selv. Dem kan du trygt ignorere.

## 1 pytest – et første eksempel

Vi følger eksempelet i [Python testing in Visual Studio Code](https://code.visualstudio.com/docs/python/testing), med programmet og testene i samme mappe.

**Steg 1.** Lag mappa `demo_tests` i prosjektmappa.

**Steg 2.** Lag filen `demo_tests/inc_dec.py` med koden som skal testes:

```python
def increment(x):
    return x + 1

def decrement(x):
    return x - 1
```

**Steg 3.** Lag filen `demo_tests/test_inc_dec.py` med testene:

```python
import inc_dec

def test_increment():
    assert inc_dec.increment(3) == 4

def test_decrement():
    assert inc_dec.decrement(3) == 4
```

> **Navneregler**
> * Både testfilene og testfunksjonene må starte med `test_`, ellers finner ikke pytest dem.
> * Filnavnet til en modul må bare inneholde bokstaver, tall og understrek. `import inc_dec` finner bare en fil som heter nøyaktig `inc_dec.py` – ikke `inc-dec.py`, `inc dec.py` eller `inc:dec.py`.

**Steg 4.** Kjør testene med ▷ i Testing-panelet, eller i terminalen:

```
python -m pytest demo_tests
```

**Dette skal du se:** Én test passerer og én feiler. I terminalen står det `.F` – punktum betyr passert, `F` betyr feilet. Se hvor mye informasjon vi får om feilen:

```
    def test_decrement():
>       assert inc_dec.decrement(3) == 4
E       assert 2 == 4
```

`2` er det `decrement(3)` faktisk returnerte, og `4` er det testen forventet.

**Steg 5. Hvilken er feil – koden eller testen?** Her er det testen: 3 − 1 er 2, ikke 4. Rett testen til `== 2` og kjør på nytt. Nå skal begge testene være grønne.

## 2 Testdrevet utvikling

I **testdrevet utvikling** (*TDD – test-driven development*) skriver vi testene **først**:

```mermaid
flowchart LR
    R[Skriv en test som feiler] --> G[Skriv kode så testen passerer] --> B[Forbedre koden] --> R
```

Å forbedre koden uten å endre hva den gjør, kalles **refaktorering**. TDD passer godt sammen med smidige metoder.

En vanlig mal for enhetstester er **Arrange – Act – Assert**: sett opp testmiljøet, utfør handlingen og sjekk resultatet.

### Eksempel: billettpriser

Funksjonen `billettpris(alder: int) -> int` skal returnere prisen for en reisende:

| Reisende | Alder | Pris |
|---|---|---|
| Barn/ungdom | 6–17 år | 63 |
| Voksen | 18–66 år | 157 |
| Honnør | 67 år eller mer | 79 |

Vi antar at alderen allerede er sjekket til å være et heltall mellom 6 og 130. Vi starter med testene og en «tom» funksjon. Legg merke til at vi tester en verdi midt i hvert intervall **og grenseverdiene**.

**Steg 1.** Lag mappa `billett`.

**Steg 2.** Lag testfilen `billett/test_priser.py`:

```python
import priser

def test_billettpris_barn_ungdom():
    assert priser.billettpris(6) == 63
    assert priser.billettpris(10) == 63
    assert priser.billettpris(17) == 63

def test_billettpris_voksen():
    assert priser.billettpris(18) == 157
    assert priser.billettpris(50) == 157
    assert priser.billettpris(66) == 157

def test_billettpris_honnør():
    assert priser.billettpris(67) == 79
    assert priser.billettpris(80) == 79
    assert priser.billettpris(130) == 79
```

**Steg 3.** Lag `billett/priser.py` med en «tom» funksjon:

```python
def billettpris(alder: int) -> int:
    return
```

**Steg 4.** Kjør testene.

**Dette skal du se:** Alle tre feiler. Det er helt riktig – i TDD skal testene feile før vi har skrevet koden. Funksjonen returnerer `None`, så feilmeldingene sier for eksempel `assert None == 63`.

Nå gjør vi **én test grønn om gangen**.

**Steg 5 – honnør.** Endre `priser.py` til:

```python
def billettpris(alder: int) -> int:
    if alder > 66: return 79
```

Kjør testene. **Dette skal du se:** Honnør-testen passerer. De to andre feiler fortsatt, fordi funksjonen returnerer `None` for alle andre aldre.

**Steg 6 – voksen.** Legg til en linje:

```python
def billettpris(alder: int) -> int:
    if alder > 66: return 79
    if 17 < alder < 67: return 157
```

Kjør testene. **Dette skal du se:** To av tre passerer.

**Steg 7 – barn/ungdom.** Det er bare de som er igjen, så her trenger vi ingen betingelse:

```python
def billettpris(alder: int) -> int:
    if alder > 66: return 79
    if 17 < alder < 67: return 157
    return 63
```

Kjør testene. **Dette skal du se:** Alle tre passerer.

### Refaktorering

**Kan koden forbedres?** Når én betingelse er oppfylt, trenger vi ikke sjekke de neste. Med `if`–`elif`–`else` i logisk rekkefølge og kommentarer blir aldersgrensene tydelige og lette å sammenligne med tabellen.

**Steg 8.** Endre `priser.py` til:

```python
def billettpris(alder: int) -> int:
    if 6 <= alder <= 17:
        return 63       # barn/ungdom
    elif 18 <= alder <= 66:
        return 157      # voksen
    else:
        return 79       # honnør
```

Kjør testene. **Dette skal du se:** Testene passerer fortsatt – refaktoreringen ødela ingenting. Det er hele poenget med automatiske tester.

## 3 Parametrisering

En test kan kjøres mange ganger med ulike data med dekoratoren `@pytest.mark.parametrize`. Vi oppgir navnene på parameterne og en liste med tupler. Hver tuppel blir en egen test.

**Steg 1.** Erstatt alt i `billett/test_priser.py` med:

```python
import pytest
import priser

@pytest.mark.parametrize('alder, forventet', [
    (6, 63), (10, 63), (17, 63),
    (18, 157), (50, 159), (66, 157),
    (67, 79), (80, 79), (130, 79),
])
def test_billettpris(alder, forventet):
    assert priser.billettpris(alder) == forventet
```

**Steg 2.** Kjør testene. I Testing-panelet kan du klikke deg ned under `test_billettpris` og se hver tuppel som en egen test. I terminalen:

```
python -m pytest billett -v
```

**Dette skal du se:** Åtte tester passerer, mens `test_billettpris[50-159]` feiler. Én av tuplene har en feil med vilje. **Hvilken er feil – koden eller testen?** Rett feilen og kjør på nytt.

> I terminalen blir prosenttallene til høyre røde etter den første feilen. Fargen viser hvordan hele kjøringen har gått *så langt*, ikke hvordan den enkelte testen gikk. Se på `PASSED` og `FAILED` for å finne ut hvilke tester som feilet.

### Tre måter å teste på

Tenk deg en eksamensoppgave der du skal teste en funksjon etter en gitt testplan:

| Metode | Fordeler | Ulemper |
|---|---|---|
| `print` | enkel, rask tilbakemelding | må sjekke hver test manuelt |
| `assert` | sier ifra bare når noe er feil | ser ikke hva funksjonen faktisk returnerte, og stopper ved første feil |
| pytest med parametere | kjører alle tester, oversikt over hva som passerer og feiler, viser faktisk verdi | litt mer å sette opp |

Pytest med parametere tar ikke nødvendigvis lengre tid. Oppgave 2 lar deg prøve alle tre.

## 4 Markeringer

`parametrize` er en innebygd **markering**. Vi kan også lage egne markeringer for å gruppere tester: med `@pytest.mark.treg` over noen tester kjører `pytest -m treg` bare dem. Noen innebygde:

| Markering | Virkning |
|---|---|
| `@pytest.mark.skip(reason='…')` | testen kjøres ikke |
| `@pytest.mark.skipif(sys.version_info < (3, 10), reason='…')` | kjøres bare på visse vilkår |
| `@pytest.mark.xfail` | testen *forventes* å feile |

**Steg 1.** Lag filen `billett/test_markeringer.py`:

```python
import sys
import pytest
import priser

@pytest.mark.skip(reason='ikke ferdig ennå')
def test_familiebillett():
    assert priser.familiepris(2, 2) == 400

@pytest.mark.skipif(sys.version_info < (3, 10), reason='krever Python 3.10')
def test_match():
    match priser.billettpris(30):
        case 157: pass
        case _: assert False

@pytest.mark.xfail(reason='alder under 6 er ikke definert')
def test_småbarn():
    assert priser.billettpris(3) == 0
```

**Steg 2.** Kjør bare denne filen i terminalen:

```
python -m pytest billett/test_markeringer.py -v -rsx
```

`-rsx` gjør at pytest skriver ut *hvorfor* tester ble hoppet over (`s`) eller var forventet å feile (`x`).

**Dette skal du se:** `test_familiebillett` er `SKIPPED`, `test_match` er `PASSED` (hvis du har Python 3.10 eller nyere) og `test_småbarn` er `XFAIL`. Legg merke til at `test_familiebillett` ikke krasjer, selv om funksjonen `familiepris` ikke finnes – testen kjøres jo ikke.

## 5 Videre

Senere (3.7 Enhetstesting) lærer vi å teste **unntak**, bruke **fixtures** og **monkeypatching**:

* *Fixtures* forbereder testene med felles kode, for eksempel databasetilkoblinger eller testobjekter (kalles *setup/teardown* i andre verktøy).
* *Monkeypatching* lar oss etterligne (*mocke*) objekter, slik at vi kan teste funksjoner som avhenger av eksterne data – et tilfeldig terningkast eller et API-kall.

## 6 Oppgaver

### Oppgave 1 – BMI med TDD
Bruk testdrevet utvikling til å lage `bmi_klassifisering(høyde: float, vekt: float) -> str`. BMI = vekt / høyde², med høyde i meter og vekt i kg. Anta at argumentene allerede er sjekket.

| Klassifisering | BMI |
|---|---|
| Undervekt | 18,4 eller lavere |
| Normalvekt | 18,5–24,9 |
| Overvekt | 25,0–29,9 |
| Fedme, grad 1 | 30,0–34,9 |
| Fedme, grad 2 | 35,0–39,9 |
| Fedme, grad 3 | 40,0 eller høyere |

**Steg 1.** Lag mappa `oppgave1_bmi`.

**Steg 2.** Lag `oppgave1_bmi/test_bmi.py` og skriv testene der. Ta med grenseverdier, og bruk gjerne `parametrize`:

```python
import pytest
from bmi import bmi_klassifisering

# Skriv testene her
```

**Steg 3.** Lag `oppgave1_bmi/bmi.py` med en «tom» funksjon:

```python
def bmi_klassifisering(høyde: float, vekt: float) -> str:
    return ''
```

**Steg 4.** Kjør testene og se at alle feiler. Gjør dem så grønne én om gangen, og refaktorer til slutt.

### Oppgave 2 – Karakterer: tre måter å teste på
*Tilsvarer deler av oppgave 6 på IT2-eksamen våren 2023.*

Brukeren skriver en poengsum fra 0 til 100, og programmet skal gi karakter:

* under 50: «Ikke bestått»
* 50–69: «Bestått»
* 70–89: «Godt bestått»
* 90 eller mer: «Meget godt bestått»
* under 0 eller over 100: «Ikke gyldig poengsum!»

**Steg 1.** Lag mappa `oppgave2_karakter` og filen `oppgave2_karakter/karakter.py` med funksjonen under. Den har feil med vilje.

```python
def karakter(poengsum: int) -> str:
    if poengsum < 50:
        return 'Ikke bestått'
    elif 50 < poengsum < 69:
        return 'Bestått'
    elif 70 < poengsum < 89:
        return 'Godt bestått'
    elif 90 < poengsum < 100:
        return 'Meget godt bestått'
    else:
        return 'Ikke gyldig poengsum!'
```

**Steg 2.** Test funksjonen etter testplanen på tre måter, og fyll inn de faktiske resultatene i tabellen under. Ta tiden på hver deloppgave.

a) **`print`:** Lag `oppgave2_karakter/sjekk_print.py`. Start med `from karakter import karakter` og skriv én `print` per test, for eksempel `print(30, karakter(30))`. Kjør filen med ▷ (*Run Python File*) øverst til høyre.

b) **`assert`:** Lag `oppgave2_karakter/sjekk_assert.py` med én `assert` per test, for eksempel `assert karakter(30) == 'Ikke bestått'`. Kjør filen på samme måte. Hva skjer ved første feil?

c) **pytest med parametere:** Lag `oppgave2_karakter/test_karakter.py` med `@pytest.mark.parametrize`, slik som i del 3. Kjør testene i Testing-panelet.

> Filene i a) og b) heter `sjekk_…` og ikke `test_…`, så pytest ikke prøver å kjøre dem.

| Test | Poengsum | Forventet | Faktisk |
|---|---|---|---|
| 1 | 30 | Ikke bestått | |
| 2 | 65 | Bestått | |
| 3 | 82 | Godt bestått | |
| 4 | 97 | Meget godt bestått | |
| 5 | 102 | Ikke gyldig poengsum! | |
| 6 | 0 | Ikke bestått | |
| 7 | 50 | Bestått | |
| 8 | 69 | Bestått | |
| 9 | 70 | Godt bestått | |
| 10 | 89 | Godt bestått | |
| 11 | 90 | Meget godt bestått | |
| 12 | 100 | Meget godt bestått | |
| 13 | −1 | Ikke gyldig poengsum! | |

Hvilken metode var raskest? Hvilken ga best oversikt? Rett til slutt feilene i `karakter`, og kjør pytest-testene på nytt til alle er grønne.